# GSS Data Processing

This notebook creates a subset of the General Social Survey (GSS) data for analysis.
It extracts selected variables from the 2022 survey year.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

In [2]:
DATA_DIR = Path("../data")
INPUT_FILE = DATA_DIR / "GSS_stata" / "gss7224_r2.parquet"
OUTPUT_FILE = DATA_DIR / "gss_2022.csv"

YEAR = 2022

VARIABLES = [
    "year",
    "id",
    "age",
    "sex",
    "race",
    "relig",  # Religious affiliation
    "degree",
    "satjob",
    "hlthdep",
    # "feeldown",
    # "nointerest",
    "stress",
    "feelnerv",
    "worry",
    "wrkmeangfl",
    "richwork",
    "satfin",
    "finrela",
    "lifenow",
    "wrkstat",
    "chngtme",  # Flexibility in work hours
    "hrs1",
    "hrs2",
    "spocc10",
    "spind10",
    "occ10", # Occupation 
    "realrinc" # Respondant income in intflation-adjusted dollars
]

In [3]:
gss_full = pd.read_parquet(INPUT_FILE)
print(f"Full dataset: {gss_full.shape[0]:,} rows, {gss_full.shape[1]} columns")

Full dataset: 75,699 rows, 6904 columns


In [4]:
gss_subset = gss_full.query("year == @YEAR")[VARIABLES].copy()
print(f"Subset: {gss_subset.shape[0]:,} rows, {gss_subset.shape[1]} columns")

Subset: 3,544 rows, 25 columns


In [5]:
gss_subset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3544 entries, 68846 to 72389
Data columns (total 25 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   year        3544 non-null   int16  
 1   id          3544 non-null   int16  
 2   age         3336 non-null   float64
 3   sex         3524 non-null   float64
 4   race        3491 non-null   float64
 5   relig       3483 non-null   float64
 6   degree      3544 non-null   float64
 7   satjob      2463 non-null   float64
 8   hlthdep     1127 non-null   float64
 9   stress      1953 non-null   float64
 10  feelnerv    1942 non-null   float64
 11  worry       1939 non-null   float64
 12  wrkmeangfl  1953 non-null   float64
 13  richwork    1452 non-null   float64
 14  satfin      3526 non-null   float64
 15  finrela     3500 non-null   float64
 16  lifenow     1780 non-null   float64
 17  wrkstat     3536 non-null   float64
 18  chngtme     1960 non-null   float64
 19  hrs1        1938 non-null  

## Ordinal Variable Re-encoding

Standardize ordinal variables so higher values consistently mean MORE of the construct measured:
- **satjob**: 1=Very Dissatisfied → 4=Very Satisfied (reversed from original)
- **satfin**: 1=Not Satisfied → 3=Pretty Well Satisfied (reversed from original)
- **wrkmeangfl**: 1=Not At All Meaningful → 4=Very Meaningful (reversed from original)
- **chngtme**: 1=Not At All Flexible → 4=Very Flexible (reversed from original)
- **stress**: 1=Never → 5=Always (reversed from original "How often do you find your work stressful?" where 1=Always, 5=Never)

Variables already following "higher = more" convention (unchanged):
- hlthdep, feelnerv, worry: higher = more distress (1=None of the time → more = All of the time)
- lifenow, finrela, degree: higher = more of construct

In [6]:
# Re-encode ordinal variables so higher values = MORE of the construct
# Reversal formula: new_value = (K + 1) - old_value

# Satisfaction variables (originally: 1=most satisfied, now: higher=more satisfied)
gss_subset["satjob"] = 5 - gss_subset["satjob"]          # 1-4 → 4-1
gss_subset["satfin"] = 4 - gss_subset["satfin"]          # 1-3 → 3-1

# Work variables (originally: 1=most, now: higher=more)
gss_subset["wrkmeangfl"] = 5 - gss_subset["wrkmeangfl"]  # 1-4 → 4-1
gss_subset["chngtme"] = 5 - gss_subset["chngtme"]        # 1-4 → 4-1

# Stress (originally: 1=Always, 5=Never; now: higher=more stress)
gss_subset["stress"] = 6 - gss_subset["stress"]          # 1-5 → 5-1

print("Ordinal variables re-encoded (higher = more):")
for var in ["satjob", "satfin", "wrkmeangfl", "chngtme", "stress"]:
    vals = sorted(gss_subset[var].dropna().unique())
    print(f"  {var}: {vals}")

Ordinal variables re-encoded (higher = more):
  satjob: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
  satfin: [np.float64(1.0), np.float64(2.0), np.float64(3.0)]
  wrkmeangfl: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
  chngtme: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
  stress: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]


In [7]:
gss_subset["realrinc"]

68846    40900.0
68847        NaN
68848    18405.0
68849     2249.5
68850        NaN
          ...   
72385        NaN
72386    27607.5
72387    33742.5
72388    27607.5
72389        NaN
Name: realrinc, Length: 3544, dtype: float64

In [8]:
missing = gss_subset.isna().sum()
missing_pct = (missing / len(gss_subset) * 100).round(1)
missing_df = pd.DataFrame({"Missing": missing, "% Missing": missing_pct})
print("Missing values per variable:")
missing_df.sort_values(by="% Missing", ascending=False)

Missing values per variable:


,Missing,% Missing
hrs2,3459,97.6
hlthdep,2417,68.2
spind10,2187,61.7
spocc10,2184,61.6
richwork,2092,59.0
lifenow,1764,49.8
hrs1,1606,45.3
worry,1605,45.3
feelnerv,1602,45.2
stress,1591,44.9


## Exploring FEELNERV and WORRY

These two variables measure related anxiety constructs and may be candidates for combining.

In [9]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("FEELNERV", "WORRY"))

feelnerv_counts = gss_subset["feelnerv"].value_counts().sort_index()
worry_counts = gss_subset["worry"].value_counts().sort_index()

fig.add_trace(
    go.Bar(x=feelnerv_counts.index, y=feelnerv_counts.values, name="FEELNERV"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=worry_counts.index, y=worry_counts.values, name="WORRY"),
    row=1, col=2
)

fig.update_layout(
    title_text="Distribution of FEELNERV and WORRY",
    showlegend=False,
    height=400
)
fig.update_xaxes(title_text="Response", row=1, col=1)
fig.update_xaxes(title_text="Response", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.show()

In [10]:
crosstab = pd.crosstab(gss_subset["feelnerv"], gss_subset["worry"])

fig = px.imshow(
    crosstab,
    labels=dict(x="WORRY", y="FEELNERV", color="Count"),
    title="Joint Distribution of FEELNERV and WORRY",
    text_auto=True,
    color_continuous_scale="Blues"
)
fig.update_layout(height=500, width=600)
fig.show()

In [11]:
from scipy.stats import spearmanr, kendalltau

complete_cases = gss_subset[["feelnerv", "worry"]].dropna()
n = len(complete_cases)

pearson_corr = complete_cases["feelnerv"].corr(complete_cases["worry"])
spearman_corr, spearman_p = spearmanr(complete_cases["feelnerv"], complete_cases["worry"])
kendall_corr, kendall_p = kendalltau(complete_cases["feelnerv"], complete_cases["worry"])

print(f"Correlation between FEELNERV and WORRY (n={n:,}):")
print(f"  Pearson r:  {pearson_corr:.3f}")
print(f"  Spearman ρ: {spearman_corr:.3f} (p={spearman_p:.2e})")
print(f"  Kendall τ:  {kendall_corr:.3f} (p={kendall_p:.2e})")

Correlation between FEELNERV and WORRY (n=1,933):
  Pearson r:  0.707
  Spearman ρ: 0.692 (p=4.65e-275)
  Kendall τ:  0.656 (p=2.15e-223)


## Create Anxiety Composite

Given the high correlation between FEELNERV and WORRY, we combine them into a single anxiety score using the mean.

In [12]:
# Create anxiety as mean of feelnerv and worry (uses available value if one is missing)
gss_subset["anxiety"] = gss_subset[["feelnerv", "worry"]].mean(axis=1)

print(f"Anxiety variable created:")
print(f"  Non-null values: {gss_subset['anxiety'].notna().sum():,}")
print(f"  Range: {gss_subset['anxiety'].min():.1f} - {gss_subset['anxiety'].max():.1f}")
print(f"  Mean: {gss_subset['anxiety'].mean():.2f}")
print(f"  Median: {gss_subset['anxiety'].median():.1f}")

Anxiety variable created:
  Non-null values: 1,948
  Range: 1.0 - 4.0
  Mean: 1.67
  Median: 1.5


## Create Hours Worked Variable

GSS has two hours variables that capture different populations via skip logic:
- **hrs1**: "Hours worked last week, at all jobs" — asked to current workers (wrkstat 1-2)
- **hrs2**: "Hours usually work per week" — asked to temporarily not working (wrkstat 3)

These variables are mutually exclusive (zero overlap), so we combine them using coalesce logic.
Non-workers (retired, unemployed, students, etc.) correctly have no hours data.


In [13]:
# Combine hrs1 and hrs2 into single hours_worked variable
# hrs1 takes precedence (current workers), hrs2 used for temporarily not working
gss_subset["hours_worked"] = gss_subset["hrs1"].combine_first(gss_subset["hrs2"])

print("Hours worked variable created:")
print(f"  Non-null values: {gss_subset['hours_worked'].notna().sum():,}")
print(f"  Range: {gss_subset['hours_worked'].min():.0f} - {gss_subset['hours_worked'].max():.0f}")
print(f"  Mean: {gss_subset['hours_worked'].mean():.1f}")
print(f"  Median: {gss_subset['hours_worked'].median():.0f}")

# Coverage by work status
print("\nCoverage by work status:")
wrkstat_labels = {1: "Full-time", 2: "Part-time", 3: "Temp not working", 
                  4: "Unemployed", 5: "Retired", 6: "School", 7: "Keeping house", 8: "Other"}
for val in sorted(gss_subset["wrkstat"].dropna().unique()):
    subset = gss_subset[gss_subset["wrkstat"] == val]
    avail = gss_subset.loc[subset.index, "hours_worked"].notna().sum()
    label = wrkstat_labels.get(int(val), "Unknown")
    print(f"  {label}: {avail}/{len(subset)} ({100*avail/len(subset):.0f}%)")


Hours worked variable created:
  Non-null values: 2,023
  Range: 0 - 89
  Mean: 40.2
  Median: 40

Coverage by work status:
  Full-time: 1594/1599 (100%)
  Part-time: 344/347 (99%)
  Temp not working: 85/90 (94%)
  Unemployed: 0/184 (0%)
  Retired: 0/773 (0%)
  School: 0/101 (0%)
  Keeping house: 0/278 (0%)
  Other: 0/164 (0%)


In [14]:
gss_subset.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to {OUTPUT_FILE}")

Saved to ../data/gss_2022.csv
